# pySLAM (SIFT, monocular) vs custom SlamWorker -- KITTI seq00 benchmark

Runs [luigifreda/pyslam](https://github.com/luigifreda/pyslam)'s **monocular** educational VO
(`VisualOdometryEducational`, the same class `main_vo.py` uses) with the **SIFT** feature tracker on
KITTI sequence 00, headless, then computes ATE (2D Umeyama-aligned RMSE, same methodology as this
project's own `analyze/kitti_ate.cpp`) against `poses/00.txt` -- a real reference point for the custom
`SlamWorker` pipeline this session has been optimizing.

**Monocular, not stereo**: `VisualOdometryEducational` is pySLAM's lightweight classic 2-view VO --
it needs ground truth at each step to recover scale (monocular has none of its own), same as this
project's own vision-only baseline. This is a deliberate simplification -- pySLAM's stereo path only
exists inside the full `main_slam.py` SLAM stack (backend BA, loop closing, g2o), which is unnecessary
weight for a plain VO comparison.

**Important caveat, confirmed by reading `visual_odometry.py` directly**: this class reads the REAL
ground-truth translation scale at EVERY frame (`kUseGroundTruthScale = True`) to compose
$C_k = C_{k-1} \cdot [R, s\,t]$ -- it never triangulates or infers scale itself, unlike this project's
own `SlamWorker`. So a low ATE here is **not** purely "SIFT tracking is more accurate" -- part of it is
simply being handed real-world scale continuously, something `SlamWorker` never gets. This result is
only a fair comparison for the **rotation/direction** accuracy (from the Essential Matrix estimate,
5-point Nister + RANSAC), not the scale dimension. Keep that in mind when interpreting the final number.

**Why a custom driver instead of just running `main_vo.py` directly**: that script is written for
interactive local use (`cv2.imshow`, matplotlib/Rerun/Pangolin viewers, a `while True` loop reading
keypresses) with no headless flag -- it will hang or error on a display-less Kaggle kernel. The driver
cell below imports the exact same VO class/factories `main_vo.py` uses and just drops every visualization
call, looping until the dataset is exhausted.

**Before running**: attach a KITTI odometry (sequence 00) Kaggle Dataset (`image_0` grayscale is fine)
so it's visible under `/kaggle/input`, and enable internet access in the notebook's settings (needed to
`git clone` and `pip install`).

In [ ]:
# --- [1/6] Clone pySLAM (recursive: it vendors a couple of thirdparty submodules) ---
%cd /kaggle/working
!git clone --recursive https://github.com/luigifreda/pyslam.git
%cd /kaggle/working/pyslam

In [ ]:
# --- [2/6] Locate the attached KITTI dataset (grayscale image_0 + poses/00.txt) ---
import glob, os

hits = glob.glob("/kaggle/input/**/image_0", recursive=True)
assert hits, "No image_0/ folder found under /kaggle/input -- attach a KITTI odometry (seq00) dataset"
image_dir = hits[0]

sequence_dir = os.path.dirname(image_dir)  # .../sequences/00
dataset_base = os.path.dirname(os.path.dirname(sequence_dir))  # parent of 'sequences'

poses_hits = glob.glob("/kaggle/input/**/poses/00.txt", recursive=True)
assert poses_hits, "No poses/00.txt found under /kaggle/input -- needed for ATE against ground truth"
poses_path = poses_hits[0]

print("dataset_base (KITTI_DATASET.base_path):", dataset_base)
print("sequence_dir:", sequence_dir)
print("poses_path:", poses_path)

In [ ]:
# --- [3/6] Install dependencies ---
# pySLAM has no lighter "extras" install path (single pyproject.toml, no
# optional-dependencies groups) -- even a SIFT-only monocular run pulls in
# its full dependency list (it also supports many deep-learning feature
# types this run won't use) AND needs its compiled pybind11 C++ core
# (pyslam_utils -- feature_manager.py imports it unconditionally, not behind
# a try/except, so there is no way to skip build_cpp_core.sh even here).
# Confirmed by reading the source directly (2026-07-21) before writing this
# notebook, not assumed. Still the single slowest step here.
!chmod +x install_all.sh pyenv-create.sh pyenv-activate.sh build_cpp_core.sh 2>/dev/null
!./install_all.sh

In [ ]:
# --- [4/6] Configure config.yaml for KITTI seq00, monocular ---
import yaml

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["DATASET"]["type"] = "KITTI_DATASET"
cfg["KITTI_DATASET"]["type"] = "kitti"
cfg["KITTI_DATASET"]["sensor_type"] = "mono"
cfg["KITTI_DATASET"]["base_path"] = dataset_base
cfg["KITTI_DATASET"]["name"] = "00"
cfg["KITTI_DATASET"]["settings"] = "settings/KITTI00-02.yaml"
cfg["KITTI_DATASET"]["is_color"] = False
cfg["KITTI_DATASET"]["groundtruth_file"] = "auto"
cfg["KITTI_DATASET"]["start_frame_id"] = 0

with open("config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, default_flow_style=False)

print(yaml.safe_dump({"KITTI_DATASET": cfg["KITTI_DATASET"]}))

In [ ]:
# --- [5/6] Headless monocular SIFT VO driver ---
# Same objects main_vo.py wires together (pyslam.config.Config,
# dataset_factory, groundtruth_factory, PinholeCamera,
# feature_tracker_factory + FeatureTrackerConfigs.SIFT, VisualOdometryEducational)
# but with every cv2.imshow/matplotlib/Rerun/Pangolin call removed -- those
# assume an interactive display this Kaggle kernel doesn't have.
%%writefile run_headless_mono_sift.py
import sys
from pyslam.config import Config
from pyslam.slam.visual_odometry import VisualOdometryEducational
from pyslam.slam import PinholeCamera
from pyslam.io.ground_truth import groundtruth_factory
from pyslam.io.dataset_factory import dataset_factory
from pyslam.local_features.feature_tracker import feature_tracker_factory
from pyslam.local_features.feature_tracker_configs import FeatureTrackerConfigs

config = Config()
dataset = dataset_factory(config)
groundtruth = groundtruth_factory(config.dataset_settings)
cam = PinholeCamera(config)

num_features = config.num_features_to_extract if config.num_features_to_extract > 0 else 2000
tracker_config = FeatureTrackerConfigs.SIFT
tracker_config["num_features"] = num_features
feature_tracker = feature_tracker_factory(**tracker_config)

vo = VisualOdometryEducational(cam, groundtruth, feature_tracker)

img_id = 0
while dataset.is_ok:
    img = dataset.getImageColor(img_id)
    if img is None:
        break
    timestamp = dataset.getTimestamp()
    vo.track(img, None, None, img_id, timestamp)
    if img_id % 200 == 0:
        print(f"frame {img_id}, matches={vo.num_matched_kps}, inliers={vo.num_inliers}", file=sys.stderr)
    img_id += 1

print(f"done: {img_id} frames processed, {len(vo.traj3d_est)} trajectory points", file=sys.stderr)

with open("results/trajectory_mono_sift.txt", "w") as f:
    for x, y, z in vo.traj3d_est:
        f.write(f"{x} {y} {z}\n")

In [ ]:
!mkdir -p results
!bash -lc "source pyenv-activate.sh && python run_headless_mono_sift.py"

## ATE evaluation

Same methodology as this project's own `analyze/kitti_ate.cpp`: 2D (X, Z) Umeyama alignment
(rotation + scale + translation) fit to ground truth, then RMSE of the aligned trajectory -- so this
number is directly comparable to this session's own vision-only monocular baseline (146.307m,
SIFT+P3P, no BA/loop-closure) as well as the best full-pipeline result (~125m, with continuous BA +
loop closure + a stereo scale anchor -- pySLAM's mono VO here has none of those, so expect it to land
closer to the 146m reference than the 125m one).

In [ ]:
import numpy as np

def load_traj_xz(path):
    pts = []
    with open(path) as f:
        for line in f:
            vals = [float(v) for v in line.split()]
            if len(vals) < 3:
                continue
            x, z = vals[0], vals[2]  # vo.traj3d_est entries are already plain (x, y, z)
            pts.append((x, z))
    return np.array(pts)

def load_kitti_gt_xz(path):
    pts = []
    with open(path) as f:
        for line in f:
            vals = [float(v) for v in line.split()]
            if len(vals) < 12:
                continue
            x, z = vals[3], vals[11]  # translation column of the 3x4 pose matrix
            pts.append((x, z))
    return np.array(pts)

def umeyama_2d(src, dst):
    # Same closed-form similarity fit (rotation + isotropic scale + translation)
    # this project's own kitti_ate.cpp uses for ATE alignment.
    n = min(len(src), len(dst))
    src, dst = src[:n], dst[:n]
    mu_src, mu_dst = src.mean(axis=0), dst.mean(axis=0)
    src_c, dst_c = src - mu_src, dst - mu_dst
    cov = (dst_c.T @ src_c) / n
    U, S, Vt = np.linalg.svd(cov)
    d = np.sign(np.linalg.det(U @ Vt))
    D = np.diag([1, d])
    R = U @ D @ Vt
    var_src = (src_c ** 2).sum() / n
    scale = np.trace(np.diag(S) @ D) / var_src
    t = mu_dst - scale * R @ mu_src
    aligned = (scale * (R @ src.T)).T + t
    rmse = np.sqrt(((aligned - dst[:n]) ** 2).sum(axis=1).mean())
    return rmse, scale, aligned

traj = load_traj_xz("results/trajectory_mono_sift.txt")
gt = load_kitti_gt_xz(poses_path)
print(f"trajectory points: {len(traj)}, ground-truth points: {len(gt)}")

rmse, scale, aligned = umeyama_2d(traj, gt)
path_len = np.linalg.norm(np.diff(gt, axis=0), axis=1).sum()
print(f"Recovered scale: {scale:.4f}")
print(f"GT path length:  {path_len:.1f} m")
print(f"ATE RMSE:        {rmse:.3f} m")
print(f"ATE RMSE / path: {100.0 * rmse / path_len:.2f}%")